In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/gold-helper

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
#reading data from silver circutes table and races table

df_circuits = (spark.table(f"{catalog_name}.{silver_schema}.circuits").filter(F.col('batch_id') == v_batch_id))
df_races = (spark.table(f"{catalog_name}.{silver_schema}.races").filter(F.col('batch_id') == v_batch_id))

In [0]:
df_dim_races=(
    df_races.join(df_circuits, df_races.circuit_id == df_circuits.circuit_id,"inner")
    .select(
        df_races.season.alias("season"),
        df_races.round.alias("round"),
        df_races.race_name.alias("race_name"),
        df_races.race_date.alias("race_date"),
        df_circuits.circuit_name.alias("circuit_name"),
        df_circuits.locality.alias("locality"),
        df_circuits.country.alias("country")
        ))

In [0]:
write_to_gold(input_df=df_dim_races, 
              target_table=target_table,
              merge_condition='t.season = s.season AND t.round = s.round',
              columns_to_update=[
                  'race_name',
                  'race_date',
                  'circuit_name',
                  'locality',
                  'country'
              ]
)